<a href="https://colab.research.google.com/github/polly-esp/Machine-Learning/blob/main/TRM_assignment3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain langchain-core langchain-openai langchain-chroma

In [ ]:
!unzip chroma.zip

unzip:  cannot find or open chroma.zip, chroma.zip.zip or chroma.zip.ZIP.


In [ ]:
#import and tokonize txt about olympics
import nltk
from nltk.tokenize import sent_tokenize

nltk.download("punkt_tab")

file_path = 'olympic_doc.txt'

try:
    with open(file_path, 'r', encoding='utf-8') as file:
        txt = file.read()
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found.")
    txt = ""

sentences = sent_tokenize(txt)
print(f"There are {len(sentences)} sentences.")

There are 4 sentences.


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
sentences

['\ufeffMikhail Shaidorov made history by winning the first-ever Olympic gold medal in figure skating for Kazakhstan.',
 "Alysa Liu (USA) made a triumphant comeback to win the women's title, ahead of Japan's Kaori Sakamoto.",
 'Laurence Fournier Beaudry and Guillaume Cizeron secured the gold for France in a highly competitive Ice Dance field.',
 'Team USA defended their title in the Team Event, led by strong performances from Ilia Malinin and Alysa Liu.']

In [ ]:
## DO NOT CHANGE THESE VALUES
LLM_LITE_API_BASE = "https://llmproxy.uva.nl"
LLM_EMBEDDING = "text-embedding-3-large"

In [ ]:
# YOU CAN CHANGE THESE VALUES
LLM_LITE_API_KEY = "sk-qYH2wBNhppXLIok7oIUl3w"
LLM_MODEL = "gpt-4.1-nano" # ['gpt-4o', 'gpt-4.1-nano', 'gpt-4.1', 'gpt-4o-mini', 'gpt-5', 'o3-mini', 'gpt-5.1']


In [ ]:
#finding location of chroma storage
import chromadb
client = chromadb.PersistentClient(path='./chroma')

collection = client.list_collections()
for col in collection:
  print(col.name)

cahiers-chunk-08


In [ ]:
from langchain_openai import ChatOpenAI

#create a model
llm = ChatOpenAI(
    openai_api_base=LLM_LITE_API_BASE,
    api_key=LLM_LITE_API_KEY,
    model=LLM_MODEL,
    temperature=0.1
)

from langchain_openai.embeddings import OpenAIEmbeddings

#creale a thing that do embeddings (for txt file only since chroma is already embedded)
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-large",
    openai_api_base=LLM_LITE_API_BASE,
    api_key=LLM_LITE_API_KEY
)

from langchain_chroma import Chroma

# This will create a folder "chroma" if it does not exist, or read the contents if it exists
store = Chroma(
    collection_name="cahiers-chunk-08",
    embedding_function=embeddings,
    persist_directory="./chroma",
)

from langchain_core.documents import Document
#adding txt file to model's context
store.add_documents([Document(page_content=c) for c in sentences])

retriever = store.as_retriever(search_type="similarity")

In [ ]:
query = (
    "is matthias good at cooking?\n\n"
)

docs = retriever.invoke(query)

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
prompt = 'You are English speaking football expert. When answering the question always mention the article you used to find the answer. If you do not know the answer say "I do not know"'

messages = [
    SystemMessage(content=prompt),
    HumanMessage(content="Here are the documents: " + "\n\n".join(doc.page_content for doc in docs)),
    HumanMessage(content="Here are the instructions: " + query)
]

In [ ]:
answer = llm.invoke(messages)
print(answer.content)

I do not know. The documents provided do not contain any information about Matthias or his cooking abilities.
